In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import seaborn as sns
sns.set_palette('pastel')
sns.set_style('whitegrid')
import nibabel as nib
import os
from sklearn.metrics import pairwise_distances
import statsmodels.api as sm
from statsmodels.formula.api import ols
from sklearn.metrics import silhouette_score, pairwise_distances, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score
from networkx.algorithms.community import louvain_communities
import networkx as nx
from infomap import Infomap
from scipy.stats import mode

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Paths

To ABCD tabulated data directory (ABCD_DATA_DIR), the functional connectivity data (FC_PATH; pMTG to each of 14 large-scale resting-state networks for the ABCD participants), and the output folder (OUTPUT_DIR).

In [ ]:
ABCD_DATA_DIR = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Data/abcd-data-release-5.1/core'
FC_PATH = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/abcd_pMTG_FC_data_midb61_meanFC.csv'
OUTPUT_DIR = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final'

# Utilities

Functions to assist with unsupervised clustering based on pMTG-network FC data and plotting the solutions. 

In [ ]:
def silhouette_kmeans(X, n_clusters, metric='correlation'):
    kmeans = KMeans(n_clusters=n_clusters)
    labels = kmeans.fit_predict(X)
    return silhouette_score(X, labels, metric=metric)

def add_infomap_community_assignments(df, community_dict, i=2):
    """
    Add Infomap community assignments to the DataFrame.

    Parameters:
    - df: DataFrame with 'src_subject_id' column
    - community_dict: dict {community_id: list of src_subject_ids}
    - i: integer to append to column name for uniqueness. Default = 2 (since the most stable solution has 2 communities)

    Returns:
    - df with new column 'infomap_community'
    """
    # Create a mapping from subject ID to community ID
    subject_to_community = {}
    for community_id, subjects in community_dict.items():
        for subject in subjects:
            subject_to_community[subject] = community_id

    # Map the communities to the DataFrame
    df[f'infomap_community_{i}'] = df['src_subject_id'].map(subject_to_community)
    
    return df

def radar_plot(df, comm_col, colors=['blue','orange','green','red','purple'],Pos=True):
    """
    Create radar plots for left and right hemisphere data across communities,
    with a consistent scale across all subplots.

    Parameters:
    - df: DataFrame with columns as features and rows as subjects
    - comm_col: Name of the column indicating community assignment
    """
    # Identify left and right hemisphere columns
    # Create labels 
    left_cols = [col for col in df.columns if '_L_' in col and col.endswith('_resid') and 'full' not in col]
    right_cols = [col for col in df.columns if '_R_' in col and col.endswith('_resid') and 'full' not in col]
    labels_left = [col.split('_L_')[0] for col in left_cols]
    labels_right = [col.split('_R_')[0] for col in right_cols]
    labels_left = [label.split('_')[-1] + label.split('_')[0] for label in labels_left]
    labels_right = [label.split('_')[-1] + label.split('_')[0] for label in labels_right]
    labels_left = [label.replace('left', 'L_') for label in labels_left]
    labels_left = [label.replace('right', 'R_') for label in labels_left]
    labels_right = [label.replace('left', 'L_') for label in labels_right]
    labels_right = [label.replace('right', 'R_') for label in labels_right]

    # Compute angles
    angles_left = np.linspace(0, 2 * np.pi, len(left_cols), endpoint=False).tolist()
    angles_left += angles_left[:1]
    angles_right = np.linspace(0, 2 * np.pi, len(right_cols), endpoint=False).tolist()
    angles_right += angles_right[:1]

    # Function to compute mean radar profile per community
    def get_mean_profile(df, columns, community_id):
        profile = df[df[comm_col] == community_id][columns].mean().tolist()
        profile += profile[:1]  # close the radar
        if Pos:
            profile = [max(0, val) for val in profile]
        return profile

    # Compute global min/max for scaling
    all_profiles_left = []
    all_profiles_right = []
    communities = df[comm_col].unique()

    for comm in communities:
        all_profiles_left.append(get_mean_profile(df, left_cols, comm)[:-1])
        all_profiles_right.append(get_mean_profile(df, right_cols, comm)[:-1])
    if Pos:
        global_min = 0
    else:
        global_min = min(np.min(all_profiles_left), np.min(all_profiles_right))
    global_max = max(np.max(all_profiles_left), np.max(all_profiles_right))

    # Plot
    fig, axs = plt.subplots(len(communities), 2, subplot_kw=dict(polar=True), figsize=(10, len(communities)*4))

    # Order communities by mean DMN_left_L_fz_resid
    dmn_left_col = 'DMN_left_L_fz_resid'
    if dmn_left_col in df.columns:
        dmn_means = df.groupby(comm_col)[dmn_left_col].mean()
        ordered_communities = dmn_means.sort_values().index.tolist()
    else:
        ordered_communities = sorted(communities, key=lambda x: int(x))
    communities = [comm for comm in ordered_communities if comm in df[comm_col].unique()]
    
    for i, comm in enumerate(communities):
        # Left hemisphere
        profile_left = get_mean_profile(df, left_cols, comm)
        ax_left = axs[i, 0]
        ax_left.plot(angles_left, profile_left, linewidth=2, color=colors[i % len(colors)])
        ax_left.fill(angles_left, profile_left, alpha=0.3, color=colors[i % len(colors)])
        ax_left.set_title(f'Community {comm+1} – Left Hemisphere', fontsize=12, weight='bold')
        ax_left.set_xticks(angles_left[:-1])
        ax_left.set_xticklabels(labels_left, fontsize=7)
        for label in ax_left.get_xticklabels():
            label.set_rotation(45)
        ax_left.set_ylim(global_min, global_max)

        # Right hemisphere
        profile_right = get_mean_profile(df, right_cols, comm)
        ax_right = axs[i, 1]
        ax_right.plot(angles_right, profile_right, linewidth=2, color=colors[i % len(colors)])
        ax_right.fill(angles_right, profile_right, alpha=0.3, color=colors[i % len(colors)])
        ax_right.set_title(f'Community {comm+1} – Right Hemisphere', fontsize=12, weight='bold')
        ax_right.set_xticks(angles_right[:-1])
        ax_right.set_xticklabels(labels_right, fontsize=7)
        for label in ax_right.get_xticklabels():
            label.set_rotation(45)
        ax_right.set_ylim(global_min, global_max)

    plt.tight_layout()
    plt.show()

def silhouette_im(data, community_dict, node_order):
    """
    Compute silhouette score given data and Infomap or Louvain community assignments.

    Parameters:
    - data: DataFrame of shape (n_subjects, n_features), indexed by subject ID
    - community_dict: dict {community_id: list of subject_ids}
    - node_order: list of subject_ids, order must match rows in `data`

    Returns:
    - silhouette score and number of communities
    """
    # Build label vector from community_dict
    subject_to_label = {
        subj: label for label, subjects in community_dict.items() for subj in subjects
    }

    # Extract data in correct order
    labels = []
    valid_subjects = []
    for subj in node_order:
        if subj in subject_to_label:
            labels.append(subject_to_label[subj])
            valid_subjects.append(subj)

    X = data.loc[valid_subjects].values
    distance_matrix = pairwise_distances(X, metric='correlation') 
    score = silhouette_score(distance_matrix, labels, metric='precomputed')
    print(f"Silhouette index solution with {len(set(labels))} communities: {score:.4f}")
    return score

def davies_bouldin_im(data, community_dict, node_order):
    """
    Compute Davies-Bouldin score given data and Infomap community assignments.

    Parameters:
    - data: DataFrame of shape (n_subjects, n_features), indexed by subject ID
    - community_dict: dict {community_id: list of subject_ids}
    - node_order: list of subject_ids, order must match rows in `data`

    Returns:
    - Davies-Bouldin score
    """
    # Build label vector from community_dict
    subject_to_label = {
        subj: label for label, subjects in community_dict.items() for subj in subjects
    }

    # Extract data in correct order
    labels = []
    valid_subjects = []
    for subj in node_order:
        if subj in subject_to_label:
            labels.append(subject_to_label[subj])
            valid_subjects.append(subj)

    if len(set(labels)) < 2:
        print("Only one community found, Davies-Bouldin index set to 10.")
        db_score = 10
    else:
        X = data.loc[valid_subjects].values
        db_score = davies_bouldin_score(X, labels)
        print(f"Davies-Bouldin index for solution with {len(set(labels))} communities: {db_score:.4f}")
    return db_score


## Infomap

In [ ]:
def find_networks(df, src_subject_ids, thresholds=[0.1, 0.2, 0.3, 0.35, 0.4, 0.43]):
    """
    Constructs a subject similarity graph using Pearson correlation of FC profiles,
    applies thresholding, and detects communities using Infomap.

    Parameters:
    - df: DataFrame of shape (n_subjects, n_features) containing FC features per subject
    - src_subject_ids: array-like of subject IDs, one per row in `df`
    - thresholds: list of percentile thresholds for edge inclusion (0–1 scale)

    Returns:
    - all_communities: dict mapping each threshold -> {community_id: list of subject IDs}
    - all_graphs: dict mapping each threshold -> networkx Graph instance
    """
    # Convert DataFrame to NumPy array and ensure subject IDs are in array format
    data = df.to_numpy()
    src_subject_ids = np.asarray(src_subject_ids)

    all_communities = {}  # Stores communities per threshold
    all_graphs = {}       # Stores graphs per threshold

    # Build mapping between numeric index and subject ID
    idx_to_subj = dict(enumerate(src_subject_ids))
    subj_to_idx = {v: k for k, v in idx_to_subj.items()}

    # Compute Pearson correlation matrix across subjects
    similarity_matrix = np.corrcoef(data)
    print(f"Similarity matrix shape: {similarity_matrix.shape}")
    n_subjects = similarity_matrix.shape[0]

    # Iterate over each threshold
    for threshold in thresholds:
        # Initialize graph with all subjects as nodes
        G = nx.Graph()
        G.add_nodes_from(src_subject_ids)

        # Extract upper triangle of similarity matrix (excluding diagonal)
        upper_tri = similarity_matrix[np.triu_indices(n_subjects, k=1)]

        # Determine correlation cutoff corresponding to given threshold percentile
        cutoff = np.percentile(upper_tri, 100 - threshold * 100)
        print(f"[Threshold {threshold:.3f}] Correlation cutoff: {cutoff:.4f}")

        # Add edges between subject pairs exceeding the correlation threshold
        for i in range(n_subjects):
            for j in range(i + 1, n_subjects):
                if similarity_matrix[i, j] > cutoff:
                    G.add_edge(src_subject_ids[i], src_subject_ids[j], weight=similarity_matrix[i, j])

        # Guard against thresholds that produce empty graphs
        if G.number_of_edges() == 0:
            raise ValueError(f"Threshold {threshold:.3f} too high — no edges remain.")

        # Initialize Infomap and add edges using integer node IDs
        im = Infomap()
        for u, v in G.edges():
            im.add_link(subj_to_idx[u], subj_to_idx[v])

        # Run Infomap community detection (with fixed seed for reproducibility)
        im.run(seed=42)

        # Extract communities and map back to original subject IDs
        communities = {}
        for node in im.nodes:
            subj_id = idx_to_subj[node.node_id]
            communities.setdefault(node.module_id, []).append(subj_id)

        # Store results for current threshold
        all_communities[threshold] = communities
        all_graphs[threshold] = G

        print(f"[Threshold {threshold:.3f}] Found {len(communities)} communities.")

        # Optional: Reorder communities by mean DMN activation, if column is present
        if 'DMN_left_L_fz_resid' in df.columns:
            # Map each subject ID to its DMN value
            dmn_col = df['DMN_left_L_fz_resid']
            subj_to_dmn = dict(zip(src_subject_ids, dmn_col))

            # Compute mean DMN value for each community
            community_means = {
                comm_id: np.mean([subj_to_dmn[subj] for subj in members])
                for comm_id, members in communities.items()
            }

            # Sort communities by increasing mean DMN value
            ordered = sorted(community_means.items(), key=lambda x: x[1])

            # Reassign community IDs based on sorted order (0, 1, 2, ...)
            new_communities = {
                i: communities[comm_id]
                for i, (comm_id, _) in enumerate(ordered)
            }

            # Replace original community structure with reordered version
            all_communities[threshold] = new_communities

    return all_communities, all_graphs


In [ ]:
df = pd.read_csv(os.path.join(OUTPUT_DIR, 'abcd_pMTG_FC_data_midb61_meanFC.csv'))
print(df)
# create a dictionary with resid column names for each network and a number
resid_columns = [col for col in df.columns if col.endswith('_resid') and "full" not in col]
fc_profile_dict = {col: i for i, col in enumerate([col for col in resid_columns])}
print('FC profile dictionary:', fc_profile_dict)
network_labels = {
    'DMN': 1, 'VAN': 7, 'Aud': 12, 'CO': 9, 'PMN': 15,
    'DAN': 5, 'FP': 3, 'PON': 16, 'Sal': 8,
    'SMd': 10, 'SMl': 11, 'Vis': 2, 'Tpole': 13, 'MTL': 14
 } 

# for every column name that contains _fz and matches a network label, add it to a fc_profile list for each subject in src_subject_id
fc_profile_left_columns = []
fc_profile_right_columns = []

for net in network_labels.keys():
    for col in df.columns:
        if f"{net}" in col and "_L_" in col:
            fc_profile_left_columns.append(col)
        if f"{net}" in col and "_R_" in col:
            fc_profile_right_columns.append(col)
fc_profile_bilateral = fc_profile_left_columns + fc_profile_right_columns
print('Bilateral FC profile columns:', fc_profile_bilateral)

### Run Infomap

In [ ]:
src_subject_ids = df['src_subject_id'].values
thresholds = [i/100 for i in range(1,51)]
communities, graphs = find_networks(df[resid_columns], src_subject_ids, thresholds)

Calculating silhouette scores

In [ ]:
s_scores = []
for threshold in thresholds:
    comms = communities[threshold]
    if len(comms) < 2:
        print(f"Threshold {threshold:.3f}: No communities found.")
        s_scores.append(None)
        continue
    # make new df with just src_subject_id and the residualized FC columns
    df_temp = df[['src_subject_id'] + resid_columns].copy()
    df_temp.set_index('src_subject_id', inplace=True)
    score = silhouette_im(df_temp, comms, src_subject_ids)
    s_scores.append(score)
    print(f"Threshold {threshold:.3f}: Silhouette score = {score:.4f}")

# Plot
plt.figure(figsize=(10, 6))
valid_thresholds = [t for t, s in zip(thresholds, s_scores) if s is not None]
valid_scores = [s for s in s_scores if s is not None]
plt.plot(valid_thresholds, valid_scores, marker='o', color='black')
plt.xlabel('Threshold')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score by Threshold')
plt.grid()
plt.show()

In [ ]:
# Calculate Davies-Bouldin score for each threshold
db_scores = []
for threshold in thresholds:
    comms = communities[threshold]
    df_temp = df[['src_subject_id'] + resid_columns].copy()
    df_temp.set_index('src_subject_id', inplace=True)
    db_score = davies_bouldin_im(df_temp, comms, src_subject_ids)
    db_scores.append(db_score)
    print(f"Threshold {threshold:.3f}: Davies-Bouldin index = {db_score:.4f}")
    

# Plot Davies-Bouldin scores
plt.figure(figsize=(10, 6))
good_db_scores = [d for d in db_scores if d != 10]  # Exclude scores set to 10
good_thresholds = [thresholds[i] for i, d in enumerate(db_scores) if d != 10]
plt.plot(good_thresholds, good_db_scores, marker='o', color='black')
plt.xlabel('Threshold')
plt.ylabel('Davies-Bouldin Index')
plt.title('Davies-Bouldin Index by Threshold')
plt.grid()
plt.show()


In [ ]:
# find threshold with lowest DBI score at silhouette score > 0.245
thresholds = np.array(thresholds)
db_scores = np.array(db_scores)
s_scores = np.array(s_scores, dtype=object)
# Only consider indices where s_scores is not None and > 0.20
valid_indices = np.array([i for i, s in enumerate(s_scores) if s is not None and s > 0.20])
if valid_indices.size == 0:
    print("No valid thresholds found with silhouette score > 0.20")
else:
    min_db_index = valid_indices[np.argmin(db_scores[valid_indices])]
    best_threshold = thresholds[min_db_index]
    print(f"Best threshold with silhouette score > 0.20: {best_threshold:.3f}")
    print(f"Associated Pearson correlation cutoff: {np.percentile(np.corrcoef(df[resid_columns].T), 100 - best_threshold * 100):.4f}")
    print(f"Minimum Davies-Bouldin score at this threshold: {db_scores[min_db_index]:.4f}")
    print(f"Silhouette score at this threshold: {s_scores[min_db_index]:.4f}")

threshold = best_threshold
community_dict = communities[threshold]
print(f"Number of communities at threshold {threshold}: {len(community_dict)}")
for community_id, subjects in community_dict.items():
    print(f"Community {community_id}: {len(subjects)} subjects")

# add infomap community assignments to the DataFrame
df.reset_index(inplace=True)
df = add_infomap_community_assignments(df, community_dict, i=2)
print(df.head())

In [ ]:
radar_plot(df, 'infomap_community_2')

In [ ]:
# add best 3 and 4 network Infomap solutions based on lowest DBI and high silhouette score
# there should be only one best solution for 3 and 4 communities
# 4/27.2026 - changed silhouette score threshold to 0.10 to find valid solutions for 3 and 4 communities using mean FC rather than mean TS FC data
thresholds = np.array(thresholds)
db_scores = np.array(db_scores)
s_scores = np.array(s_scores, dtype=object)
valid_indices_3 = np.array([i for i, s in enumerate(s_scores) if s is not None and s > 0.10 and len(communities[thresholds[i]]) == 3])
valid_indices_4 = np.array([i for i, s in enumerate(s_scores) if s is not None and s > 0.10 and len(communities[thresholds[i]]) == 4])
if valid_indices_3.size == 0:
    print("No valid thresholds found with silhouette score > 0.10 and 3 communities")
else:
    min_db_index_3 = valid_indices_3[np.argmin(db_scores[valid_indices_3])]
    best_threshold_3 = thresholds[min_db_index_3]
    print(f"Best threshold with silhouette score > 0.10 and 3 communities: {best_threshold_3:.3f}")
    print(f"Associated Pearson correlation cutoff: {np.percentile(np.corrcoef(df[resid_columns].T), 100 - best_threshold_3 * 100):.4f}")
    print(f"Minimum Davies-Bouldin score at this threshold: {db_scores[min_db_index_3]:.4f}")
    print(f"Silhouette score at this threshold: {s_scores[min_db_index_3]:.4f}")
    community_dict_3 = communities[best_threshold_3]
    df = add_infomap_community_assignments(df, community_dict_3, i=3)
    radar_plot(df, 'infomap_community_3')
if valid_indices_4.size == 0:
    print("No valid thresholds found with silhouette score > 0.10 and 4 communities")
else:
    min_db_index_4 = valid_indices_4[np.argmin(db_scores[valid_indices_4])]
    best_threshold_4 = thresholds[min_db_index_4]
    print(f"Best threshold with silhouette score > 0.10 and 4 communities: {best_threshold_4:.3f}")
    print(f"Associated Pearson correlation cutoff: {np.percentile(np.corrcoef(df[resid_columns].T), 100 - best_threshold_4 * 100):.4f}")
    print(f"Minimum Davies-Bouldin score at this threshold: {db_scores[min_db_index_4]:.4f}")
    print(f"Silhouette score at this threshold: {s_scores[min_db_index_4]:.4f}")
    community_dict_4 = communities[best_threshold_4]
    df = add_infomap_community_assignments(df, community_dict_4, i=4)
    radar_plot(df, 'infomap_community_4')

# K-means Clustering

Clustering individuals using the KMeans algorithm with Euclidean distance as the distance metric.

In [ ]:
ns = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
kmeans_WCSS = {}
resid_columns = [col for col in df.columns if col.endswith('_resid')]
for n in ns:
    kmeans = KMeans(n_clusters=n, random_state=42)
    kmeans.fit(df[resid_columns])
    
    df[f'kmeans_{n}_labels'] = kmeans.labels_
    kmeans_WCSS[n] = kmeans.inertia_
    
    print(f'KMeans with {n} clusters done.')


In [ ]:
# plot elbow plot for kmeans WCSS
plt.figure(figsize=(8, 5))
plt.plot(list(kmeans_WCSS.keys()), list(kmeans_WCSS.values()), marker='o', color='black')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
plt.title('Elbow Method for Optimal k in KMeans')
plt.xticks(ns)
plt.grid()
plt.show()

In [ ]:
# calculate the db score for each kmeans clustering
kmeans_db_scores = {}
for n in ns:
    if f'kmeans_{n}_labels' in df.columns:
        db_score = davies_bouldin_score(df[resid_columns], df[f'kmeans_{n}_labels'])
        kmeans_db_scores[n] = db_score
        print(f"KMeans with {n} clusters: Davies-Bouldin index = {db_score:.4f}")
# plot db scores for kmeans clustering
plt.figure(figsize=(8, 5))
plt.plot(list(kmeans_db_scores.keys()), list(kmeans_db_scores.values()), marker='o', color='black')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Davies-Bouldin Index')
plt.title('Davies-Bouldin Index for KMeans Clustering')
plt.xticks(ns)
plt.grid()
plt.show()

In [ ]:
# calculate silhouette scores for each kmeans clustering
kmeans_silhouette_scores = {}
for n in ns:
    if f'kmeans_{n}_labels' in df.columns:
        distance_matrix = pairwise_distances(df[resid_columns], metric='correlation')
        score = silhouette_score(distance_matrix, df[f'kmeans_{n}_labels'], metric='precomputed')
        kmeans_silhouette_scores[n] = score
        print(f'Silhouette score for KMeans with {n} clusters: {score:.4f}')
    else:
        print(f'Column kmeans_{n}_labels not found in DataFrame.')

In [ ]:
# plot kmeans silhouette scores
plt.figure(figsize=(8, 5))
plt.plot(list(kmeans_silhouette_scores.keys()), list(kmeans_silhouette_scores.values()), marker='o', color='black')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for K-Means Clustering')
plt.xticks(ns)
plt.grid()
plt.show()

In [ ]:
# plot silhouette scores for kmeans clustering
plt.figure(figsize=(8, 5))
plt.plot(list(kmeans_silhouette_scores.keys()), list(kmeans_silhouette_scores.values()), marker='o', color='black')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for K-Means Clustering')
plt.xticks(ns)
plt.grid()
plt.show()

In [ ]:
radar_plot(df, 'kmeans_2_labels')
radar_plot(df, 'kmeans_3_labels')
radar_plot(df, 'kmeans_4_labels')

In [ ]:
# run k-means with 2 clusters 1001 times and save the cluster assignments
# then make a consensus k_means clustering
k = 2
n_runs = 1001
all_labels = np.zeros((df.shape[0], n_runs))
for run in range(n_runs):
    kmeans = KMeans(n_clusters=k, random_state=run)
    labels = kmeans.fit_predict(df[resid_columns])
    # sort labels by mean DMN_left_L_fz_resid value
    dmn_col = 'DMN_left_L_fz_resid'
    if dmn_col in df.columns:
        dmn_means = {}
        for label in np.unique(labels):
            dmn_means[label] = df.loc[labels == label, dmn_col].mean()
        sorted_labels = sorted(dmn_means, key=dmn_means.get)
        label_mapping = {old_label: new_label for new_label, old_label in enumerate(sorted_labels)}
        labels = np.array([label_mapping[label] for label in labels])
        df['kmeans_2_run_' + str(run+1)] = labels
    all_labels[:, run] = labels
    if (run + 1) % 100 == 0:
        print(f'KMeans run {run + 1}/{n_runs} done.')
# make consensus clustering by taking the mode of each row
from scipy.stats import mode
consensus_labels, _ = mode(all_labels, axis=1)
df['kmeans_2_consensus'] = consensus_labels.flatten()
print(df['kmeans_2_consensus'].value_counts())

# Louvain

In [ ]:
def find_networks_louvain(data, src_subject_ids, thresholds=[0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45], res=1, s=42):
    """
    Constructs a subject similarity graph using Pearson correlation of FC profiles,
    applies thresholding, and detects communities using Louvain.

    Parameters:
    - data: DataFrame of shape (n_subjects, n_features)
    - src_subject_ids: array-like of subject IDs, one per row in `data`
    - thresholds: list of percentile thresholds for edge inclusion

    Returns:
    - all_communities: dict mapping threshold -> {community_id: list of src_subject_id}
    - all_graphs: dict mapping threshold -> networkx Graph
    """
    
    data = data.to_numpy()
    src_subject_ids = np.asarray(src_subject_ids)
    all_communities = {}
    all_graphs = {}

    similarity_matrix = np.corrcoef(data)
    n_subjects = similarity_matrix.shape[0]

    for threshold in thresholds:
        G = nx.Graph()
        G.add_nodes_from(src_subject_ids)

        # Compute threshold cutoff
        upper_tri = similarity_matrix[np.triu_indices(n_subjects, k=1)]
        cutoff = np.percentile(upper_tri, 100 - threshold * 100)
        print(f"[Threshold {threshold:.3f}] Correlation cutoff: {cutoff:.4f}")

        # Add edges above threshold
        for i in range(n_subjects):
            for j in range(i + 1, n_subjects):
                if similarity_matrix[i, j] > cutoff:
                    G.add_edge(src_subject_ids[i], src_subject_ids[j], weight=similarity_matrix[i, j])

        if G.number_of_edges() == 0:
            raise ValueError(f"Threshold {threshold:.3f} too high — no edges remain.")

        # Run Louvain community detection
        partition = louvain_communities(G, resolution=res, seed=s)

        # Map detected communities back to subject IDs
        communities = {i: list(comm) for i, comm in enumerate(partition)}
        all_communities[threshold] = communities
        all_graphs[threshold] = G

        print(f"[Threshold {threshold:.3f}] Found {len(communities)} communities.")

        # re-number communities based on mean DMN_left_L_fz_resid
        if 'DMN_left_L_fz_resid' in df.columns:
            # Map each subject ID to its DMN value
            dmn_col = df['DMN_left_L_fz_resid']
            subj_to_dmn = dict(zip(src_subject_ids, dmn_col))

            # Compute mean DMN value for each community
            community_means = {
                comm_id: np.mean([subj_to_dmn[subj] for subj in members])
                for comm_id, members in communities.items()
            }

            # Sort communities by increasing mean DMN value
            ordered = sorted(community_means.items(), key=lambda x: x[1])

            # Reassign community IDs based on sorted order 
            new_communities = {
                i: communities[comm_id]
                for i, (comm_id, _) in enumerate(ordered)
            }

            # Replace original community structure with reordered version
            all_communities[threshold] = new_communities

    return all_communities, all_graphs

# Run Louvain community detection
ress = [0, 0.25, 0.5, 0.75, 1, 1.25, 1.5]
df.reset_index(inplace=True)
src_subject_ids = df.loc[~df[resid_columns].isnull().any(axis=1), 'src_subject_id'].reset_index(drop=True).values
# First, find the maximum silhouette score for each resolution
thresholds = [i/20 for i in range(1, 10)]  # 0.05 to 0.45 in steps of 0.05
max_silhouette_scores = {}
for r in ress:
    louvain_communities_dict, louvain_graphs = find_networks_louvain(df[resid_columns], src_subject_ids, res=r, s=42)
    s_scores = []
    for threshold in thresholds:
        comms = louvain_communities_dict[threshold]
        if len(comms) < 2:
            print(f"Threshold {threshold:.3f}: No communities found.")
            continue
        # make new df with just src_subject_id and the residualized FC columns
        df_temp = df[['src_subject_id'] + resid_columns].copy()
        df_temp.set_index('src_subject_id', inplace=True)
        score = silhouette_im(df_temp, comms, src_subject_ids)
        s_scores.append(score)
        print(f"Threshold {threshold:.3f}: Silhouette score = {score:.4f}")
# Find the resolution corresponding to the maximum silhouette score
    max_silhouette = max(s_scores)
    best_threshold_index = s_scores.index(max_silhouette)
    best_threshold = thresholds[best_threshold_index]
    print(f"Best threshold for resolution {r}: {best_threshold:.3f} with silhouette score {max_silhouette:.4f}")
    max_silhouette_scores[r] = (best_threshold, max_silhouette)

# find the optimal resolution based on the maximum silhouette score
optimal_resolution = max(max_silhouette_scores, key=lambda k: max_silhouette_scores[k][1])
optimal_threshold, optimal_silhouette = max_silhouette_scores[optimal_resolution]
print(f"Optimal resolution: {optimal_resolution}, Threshold: {optimal_threshold:.3f}, Silhouette Score: {optimal_silhouette:.4f}")



In [ ]:
# find the optimal threshold for the optimal resolution
thresholds = [i/100 for i in range(1, 51)]  # 0.01 to 0.50 in steps of 0.01
print(df[resid_columns].shape)
louvain_communities_dict, louvain_graphs = find_networks_louvain(df[resid_columns], src_subject_ids, thresholds, res=optimal_resolution, s=42)

In [ ]:
# write new function to add louvain community assignments to the DataFrame
def add_louvain_community_assignments(df, community_dict, i=1, subtypes=2):
    """
    Add Louvain community assignments to the DataFrame.

    Parameters:
    - df: DataFrame with 'src_subject_id' column
    - community_dict: dict {community_id: list of src_subject_ids}

    Returns:
    - df with new column 'louvain_community'
    """
    # Create a mapping from subject ID to community ID
    subject_to_community = {}
    for community_id, subjects in community_dict.items():
        for subject in subjects:
            subject_to_community[subject] = community_id

    # Map the communities to the DataFrame
    if subtypes == 2:
        if i == 1:
            df['louvain_community'] = df['src_subject_id'].map(subject_to_community)
        else:
            df[f'louvain_community{i}'] = df['src_subject_id'].map(subject_to_community)
    else:
        df[f'louvain_community_{subtypes}_subtypes'] = df['src_subject_id'].map(subject_to_community)
    return df

# get silhouette scores for louvain communities at each threshold
def silhouette_louvain(data, community_dict):
    """
    Compute silhouette score for Louvain communities.

    Parameters:
    - data: DataFrame of shape (n_subjects, n_features), indexed by subject ID
    - community_dict: dict {community_id: list of subject_ids}

    Returns:
    - silhouette score
    """
    from sklearn.metrics import silhouette_score
    from sklearn.metrics.pairwise import pairwise_distances

    # Build label vector from community_dict
    subject_to_label = {
        subj: label for label, subjects in community_dict.items() for subj in subjects
    }

    # Extract data in correct order
    labels = []
    valid_subjects = []
    for subj in data.index:
        if subj in subject_to_label:
            labels.append(subject_to_label[subj])
            valid_subjects.append(subj)

    if len(set(labels)) < 2:
        print("Only one community found, silhouette score cannot be calculated.")
        return -1

    X = data.loc[valid_subjects].values
    distance_matrix = pairwise_distances(X, metric='correlation')
    
    score = silhouette_score(distance_matrix, labels, metric='precomputed')
    
    return score

from sklearn.metrics import davies_bouldin_score

def davies_bouldin_louvain(data, community_dict):
    """
    Compute Davies-Bouldin score for Louvain communities.

    Parameters:
    - data: DataFrame of shape (n_subjects, n_features), indexed by subject ID
    - community_dict: dict {community_id: list of subject_ids}

    Returns:
    - Davies-Bouldin score (float) or -1 if invalid
    """
    # Build label vector from community_dict
    subject_to_label = {
        subj: label for label, subjects in community_dict.items() for subj in subjects
    }

    # Extract data and labels for subjects in the community dict
    labels = []
    valid_subjects = []
    for subj in data.index:
        if subj in subject_to_label:
            labels.append(subject_to_label[subj])
            valid_subjects.append(subj)

    # Must have at least 2 clusters, and no cluster with fewer than 2 samples
    from collections import Counter
    label_counts = Counter(labels)

    X = data.loc[valid_subjects].values
    db_score = davies_bouldin_score(X, labels)
    return db_score

In [ ]:
louvain_s_scores = []
for threshold in thresholds:
    comms = louvain_communities_dict[threshold]
    if len(comms) < 2:
        print(f"Threshold {threshold:.3f}: No communities found.")
        louvain_s_scores.append(-1)
        continue
    # make new df with just src_subject_id and the residualized FC columns
    df_temp = df[['src_subject_id'] + resid_columns].copy()
    df_temp.set_index('src_subject_id', inplace=True)
    score = silhouette_louvain(df_temp, comms)
    louvain_s_scores.append(score)
    print(f"Threshold {threshold:.3f}: Silhouette score = {score:.4f}")
# plot silhouette scores for louvain
plt.figure(figsize=(10, 6))
plt.plot(thresholds, [s for s in louvain_s_scores], marker='o', color='black')
plt.xlabel('Threshold')
plt.ylabel('Silhouette Score')
plt.title('Louvain Silhouette Score by Threshold')
plt.grid()
plt.show()

louvain_db_scores = []
for threshold in thresholds:
    comms = louvain_communities_dict[threshold]
    if len(comms) < 2:
        print(f"Threshold {threshold:.3f}: No communities found.")
        louvain_db_scores.append(-1)
        continue
    df_temp = df[['src_subject_id'] + resid_columns].copy()
    df_temp.set_index('src_subject_id', inplace=True)
    score = davies_bouldin_louvain(df_temp, comms)
    louvain_db_scores.append(score)
    print(f"Threshold {threshold:.3f}: Davies-Bouldin index = {score:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(thresholds, louvain_db_scores, marker='o', color='black')
plt.xlabel('Threshold')
plt.ylabel('Davies-Bouldin Index')
plt.title('Louvain Davies-Bouldin Index by Threshold')
plt.grid()
plt.show()

In [ ]:
# find threshold with lowest Davies-Bouldin score at silhouette score > 0.2
best_threshold = None
best_db_score = float('inf')
for threshold, db_score in zip(thresholds, louvain_db_scores):
    if db_score < best_db_score and db_score != -1:
        # Check if silhouette score is above 0.2
        silhouette_score = louvain_s_scores[thresholds.index(threshold)]
        if silhouette_score > 0.20:
            best_threshold = threshold
            best_db_score = db_score
print(f"Best threshold with silhouette score > 0.20: {best_threshold:.3f} with Davies-Bouldin score {best_db_score:.4f} and silhouette score {louvain_s_scores[thresholds.index(best_threshold)]:.4f}")
print(f"Associated Pearson correlation cutoff: {np.percentile(np.corrcoef(df[resid_columns].T), 100 - best_threshold * 100):.4f}")

In [ ]:
# run louvain 1001 times with the optimal resolution and threshold
j = 0
for j in range(1001):
    louvain_communities_dict, louvain_graphs = find_networks_louvain(df[resid_columns], src_subject_ids, thresholds=[best_threshold], res=optimal_resolution, s=j)
    print(f"Run {j+1}/1001: Found {len(louvain_communities_dict[best_threshold])} communities.")
    add_louvain_community_assignments(df, louvain_communities_dict[best_threshold], i=j+1)

In [ ]:
# consensus clustering for louvain community assignments based on mode
from scipy.stats import mode
louvain_cols = [col for col in df.columns if col.startswith('louvain_community')]
df['louvain_consensus'] = mode(df[louvain_cols], axis=1).mode.flatten()
print(df['louvain_consensus'].value_counts())

In [ ]:
radar_plot(df, 'louvain_consensus')

In [ ]:
# also get most stable 3 and 4 community solutions for louvain using the default resolution of 1.0
# find the optimal threshold for 3 and 4 communities
thresholds = [i/100 for i in range(1, 51)]  # 0.01 to 0.50 in steps of 0.01
louvain_communities_dict, louvain_graphs = find_networks_louvain(df[resid_columns], src_subject_ids, thresholds, res=1.0, s=42)

# get silhouette and db scores for each threshold
louvain_s_scores = []
louvain_db_scores = []
for threshold in thresholds:
    comms = louvain_communities_dict[threshold]
    if len(comms) < 2:
        print(f"Threshold {threshold:.3f}: No communities found.")
        louvain_s_scores.append(-1)
        louvain_db_scores.append(-1)
        continue
    # make new df with just src_subject_id and the residualized FC columns
    df_temp = df[['src_subject_id'] + resid_columns].copy()
    df_temp.set_index('src_subject_id', inplace=True)
    s_score = silhouette_louvain(df_temp, comms)
    d_score = davies_bouldin_louvain(df_temp, comms)
    louvain_s_scores.append(s_score)
    louvain_db_scores.append(d_score)
    print(f"Threshold {threshold:.3f}: Silhouette score = {s_score:.4f}, Davies-Bouldin score = {d_score:.4f}")

# find best threshold for 3 communities
valid_indices_3 = np.array([i for i, s in enumerate(louvain_s_scores) if s is not None and s > 0.15 and len(louvain_communities_dict[thresholds[i]]) == 3])
if valid_indices_3.size == 0:
    print("No valid thresholds found with silhouette score > 0.15 and 3 communities")
else:
    min_db_index_3 = valid_indices_3[np.argmin(np.array(louvain_db_scores)[valid_indices_3])]
    best_threshold_3 = thresholds[min_db_index_3]
    print(f"Best threshold with silhouette score > 0.15 and 3 communities: {best_threshold_3:.3f}")
    print(f"Associated Pearson correlation cutoff: {np.percentile(np.corrcoef(df[resid_columns].T), 100 - best_threshold_3 * 100):.4f}")
    print(f"Minimum Davies-Bouldin index at this threshold: {louvain_db_scores[min_db_index_3]:.4f}")
    print(f"Silhouette score at this threshold: {louvain_s_scores[min_db_index_3]:.4f}")
    community_dict_3 = louvain_communities_dict[best_threshold_3]
    df = add_louvain_community_assignments(df, community_dict_3, subtypes=3)

    radar_plot(df, 'louvain_community_3_subtypes')

# find best threshold for 4 communities
valid_indices_4 = np.array([i for i, s in enumerate(louvain_s_scores) if s is not None and s > 0.15 and len(louvain_communities_dict[thresholds[i]]) == 4])
if valid_indices_4.size == 0:
    print("No valid thresholds found with silhouette score > 0.15 and 4 communities")
else:
    min_db_index_4 = valid_indices_4[np.argmin(np.array(louvain_db_scores)[valid_indices_4])]
    best_threshold_4 = thresholds[min_db_index_4]
    print(f"Best threshold with silhouette score > 0.15 and 4 communities: {best_threshold_4:.3f}")
    print(f"Associated Pearson correlation cutoff: {np.percentile(np.corrcoef(df[resid_columns].T), 100 - best_threshold_4 * 100):.4f}")
    print(f"Minimum Davies-Bouldin index at this threshold: {louvain_db_scores[min_db_index_4]:.4f}")
    print(f"Silhouette score at this threshold: {louvain_s_scores[min_db_index_4]:.4f}")
    community_dict_4 = louvain_communities_dict[best_threshold_4]
    df = add_louvain_community_assignments(df, community_dict_4, subtypes=4)
    radar_plot(df, 'louvain_community_4_subtypes')

In [ ]:
# get the date and hour
import datetime
now = datetime.datetime.now()
date_str = now.strftime("%Y-%m-%d")
hour_str = now.strftime("%H-%M")

df.to_csv(os.path.join(OUTPUT_DIR, f'midb61_meanFC_clusters_{date_str}_{hour_str}.csv'), index=False)

In [ ]:
# ensure we call the sklearn metric functions even if their names were shadowed earlier
from sklearn.metrics import silhouette_score as skl_silhouette_score, davies_bouldin_score as skl_davies_bouldin_score, calinski_harabasz_score as skl_calinski_harabasz_score

# calculate DBI for the kmeans clustering
resid_cols = [col for col in df.columns if 'resid' in col and 'fz' in col and 'full' not in col]
X = df[resid_cols].dropna().values
kmeans_labels = df.loc[df[resid_cols].dropna().index, 'kmeans_2_consensus'].values
dbi_kmeans = skl_davies_bouldin_score(X, kmeans_labels)
print(f'Davies-Bouldin Index for KMeans clustering: {dbi_kmeans:.4f}')

# same for louvain clustering
louvain_labels = df.loc[df[resid_cols].dropna().index, 'louvain_consensus'].values
dbi_louvain = skl_davies_bouldin_score(X, louvain_labels)
print(f'Davies-Bouldin Index for Louvain clustering: {dbi_louvain:.4f}')

# same for infomap clustering
infomap_labels = df.loc[df[resid_cols].dropna().index, 'infomap_community_2'].values
dbi_infomap = skl_davies_bouldin_score(X, infomap_labels)
print(f'Davies-Bouldin Index for Infomap clustering: {dbi_infomap:.4f}')

# calculate silhouette score for kmeans clustering
silhouette_kmeans_score = skl_silhouette_score(X, kmeans_labels, metric='correlation')
print(f'Silhouette Score for KMeans clustering: {silhouette_kmeans_score:.4f}')

# calculate silhouette score for louvain clustering
silhouette_louvain_score = skl_silhouette_score(X, louvain_labels, metric='correlation')
print(f'Silhouette Score for Louvain clustering: {silhouette_louvain_score:.4f}')

# calculate silhouette score for infomap clustering
silhouette_infomap_score = skl_silhouette_score(X, infomap_labels, metric='correlation')
print(f'Silhouette Score for Infomap clustering: {silhouette_infomap_score:.4f}')

# calculate calinski harabasz score for kmeans clustering
ch_kmeans = skl_calinski_harabasz_score(X, kmeans_labels)
print(f'Calinski-Harabasz Score for KMeans clustering: {ch_kmeans:.4f}')
# calculate calinski harabasz score for louvain clustering
ch_louvain = skl_calinski_harabasz_score(X, louvain_labels)
print(f'Calinski-Harabasz Score for Louvain clustering: {ch_louvain:.4f}')
# calculate calinski harabasz score for infomap clustering
ch_infomap = skl_calinski_harabasz_score(X, infomap_labels)
print(f'Calinski-Harabasz Score for Infomap clustering: {ch_infomap:.4f}')